# Seleção das Imagens com Torres dos Datasets
***
Datasets utilizados para montagem do diretório: ```TTPLA Dataset``` e ```STN-PLAD Dataset```.

### Importação dos Pacotes Utilitários
***

In [34]:
from collections import Counter
import json, os
import utils

### Seleção das Imagens com Torres: ```TTPLA Dataset```
***

In [35]:
# carregando os arquivos .json em uma lista
labels_folder = '../datasets-descompactados/TTPLA-dataset/'
jsons_files = utils.list_json_files(labels_folder)
jsons_files = sorted(jsons_files, key = utils.extract_number)

In [36]:
# contando a quantidade de objetos instanciados por classe em cada uma das imagens
count_category_counts = list()

for json_file in jsons_files:
    category_counts = utils.count_categories('../datasets-descompactados/TTPLA-dataset/' + json_file)
    count_category_counts.append(category_counts)

In [37]:
# cria uma lista com o nome das imagens que possuem torres de transmissão
images_with_towers = list()

for category in count_category_counts:
    keys = category.keys()
    for key in keys:
        count = category[key]
        if (key == 'tower_tucohy') | (key == 'tower_lattice'): 
            images_with_towers.append(category['imagePath'])
        break
        
print(f'{len(images_with_towers)} Imagens que contém torres de transmissão')

279 Imagens que contém torres de transmissão


In [38]:
# copiando para uma pasta em separado as imagens que apenas contém torres
source_path = '../datasets-descompactados/TTPLA-dataset/'
destination_path = 'TTPLA-dataset/'
utils.copy_archives(source_path, images_with_towers, destination_path)

Arquivos copiados com sucesso!


In [39]:
# pegando a lista de jsons das imagens selecionadas
jsons_with_towers = [images.replace(".jpg", ".json") for images in images_with_towers]

In [40]:
# copiando para uma pasta em separado os arquivos jsons das imagens que contém torres
utils.copy_archives(source_path, jsons_with_towers, destination_path)

Arquivos copiados com sucesso!


In [41]:
# listando os arquivos jsons das imagens selecionadas
dir_files = 'TTPLA-dataset'
extension = '.json'
list_jsons = [archive for archive in os.listdir(dir_files) if archive.endswith(extension)]

print(f'FIles .json: {len(list_jsons)} files selected.')

FIles .json: 279 files selected.


In [42]:
# tratando o arquivo json para remoção de atributos não úteis
for json_file in list_jsons:
    content_json = utils.load_json_file('TTPLA-dataset/' + json_file)
    content_json['imageData'] = 'null'

    with open('TTPLA-dataset/' + json_file, 'w') as archive:
        json.dump(content_json, archive, indent = 4)

In [43]:
# selecionando nos arquivos json apenas as classes de torres
new_shapes = list()
for json_file in list_jsons:
    content_json = utils.load_json_file('TTPLA-dataset/' + json_file)
    for shape in content_json['shapes']:
        if (shape['label'] == 'tower_tucohy') or (shape['label'] == 'tower_lattice'):
            new_shapes.append(shape)
    
    content_json['shapes'] = new_shapes
    new_shapes = list()

    with open('TTPLA-dataset/' + json_file, 'w') as archive:
        json.dump(content_json, archive, indent = 4)

In [44]:
# normalizando o nome das classes de torres de concreto e aço treliçadas
for json_file in list_jsons:
    content_json = utils.load_json_file('TTPLA-dataset/' + json_file)
    for shape in content_json['shapes']:
        shape['label'] = 'towers'
    
    with open('TTPLA-dataset/' + json_file, 'w') as archive:
        json.dump(content_json, archive, indent = 4)

### Seleção das Imagens com Torres: ```STN-PLAD Dataset```

In [5]:
# carregando os rótulos no arquivo json dentro de um dicionário
json_path = '../datasets-descompactados/STN-PLAD-dataset/annotations.json'
data = utils.load_json_file(json_path)

In [17]:
# inicializando um dicionário vazio com os IDs das imagens
images_ids = utils.initialize_images_ids_with_dict(data)

# obtendo a quantidade de instâncias de objeto por imagem
for info in data['annotations']:
    images_ids[info['image_id']].append(info['category_id'])

In [25]:
# carrega em uma lista o caminho relativo das imagens com torres
images_with_towers = list()

for index, images_instances in enumerate(images_ids.values()):
    count_aux = Counter(images_instances)
    if count_aux[0] != 0:
        images_with_towers.append(utils.image_path_search(index, data)) 

print(f'Quantidade de imagens com torres: {len(images_with_towers)} imagens')

Quantidade de imagens com torres: 131 imagens


In [29]:
# copiando para uma pasta em separado as imagens que apenas contém torres
source_path = '../datasets-descompactados/STN-PLAD-dataset/'
destination_path = 'STN-PLAD-dataset/'
utils.copy_archives(source_path, images_with_towers, destination_path)

Arquivos copiados com sucesso!


### Seleção das Imagens com Torres: ```Tower-Joint Dataset```

Esse conjunto de imagens possuem apenas imagens com torres. Entretanto, algumas imagens não serão úteis para compor o grande conjunto de dados por conter imagens que podem atrapalhar o treinamento dos algoritmos. Desse modo, uma análise visual e manual foi feita para selecionar apenas as imagens úteis. (Toda organização foi feita utilizando o software RoboFlow).